In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit, GroupKFold, learning_curve
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('mes_donnees_normalisées.csv', sep=';')
df.columns = df.columns.str.strip()

#ATTENTION : contrairement aux autres notebooks on ne remappe PAS Lt ici
#car la colonne est déjà passée en numérique dans Modele.ipynb avant l'export du csv
#(le double mapping G/R -> 0/1 mettait tout à NaN et le dropna vidait le tableau)
print(df['Lt'].unique())

#On remplace les valeurs manquantes de P1_reel et Lq_reel par la première valeur non nulle de chaque essai selon les conseils de l'encadrante
#Remarque : la règle V==0 ne marche plus sur les données normalisées (V a été standardisé donc jamais exactement 0)
#du coup on prend la première valeur dispo de l'essai, ce qui revient au même vu que P1 est constant sur un essai
p1_par_essai = df.dropna(subset=['P1_reel']).groupby('essai')['P1_reel'].first()
df['P1_reel'] = df['P1_reel'].fillna(df['essai'].map(p1_par_essai))

lq_par_essai = df.dropna(subset=['Lq_reel']).groupby('essai')['Lq_reel'].first()
df['Lq_reel'] = df['Lq_reel'].fillna(df['essai'].map(lq_par_essai))


In [ ]:
#Préparation des données

# Séparation des features et de la target
df_ut = df[['P1_reel','P2','Lt','Lq_reel','P4','P5','V','E2','P6_reel','E1']].copy()
df_ut['essai'] = df['essai']  # Ajout de la colonne 'essai' pour le GroupShuffleSplit

df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
#Remarque : CatBoost sait gérer les NaN tout seul mais on a testé les deux et le dropna donne
#de meilleurs résultats ici (les lignes incomplètes viennent d'essais difficiles qui ajoutent surtout du bruit)
print(f"Nombre de lignes après dropna(): {df_ut.shape[0]}")
print(f"Nombre d'essais : {df_ut['essai'].nunique()}")

X1 = df_ut.drop(columns=['E1'])
y1 = df_ut[['essai','E1']]

# Split par groupe pour que les lignes d'un même essai ne soient jamais des deux côtés
groupes = X1['essai']
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X1, y1, groups=groupes))

X_train, X_test = X1.iloc[train_idx].copy(), X1.iloc[test_idx].copy()
y_train, y_test = y1.iloc[train_idx].copy(), y1.iloc[test_idx].copy()
groupes_train = groupes.iloc[train_idx]

# On retire l'essai pour ne pas tricher
X_train = X_train.drop(columns=['essai'])
X_test = X_test.drop(columns=['essai'])
y_train = y_train['E1']
y_test = y_test['E1']


In [ ]:
#Recherche des meilleurs hyperparamètres pour le CatBoost sur E1
#On a comparé une dizaine de modèles (random forest, extra trees, gradient boosting, LightGBM,
#SVR, KNN, MLP, gaussian process, modèles linéaires...) : CatBoost sort premier à la fois
#en R2 sur le test ET en validation croisée groupée, là où le random forest est très instable
#d'un essai à l'autre

param_grid = {
    'depth': [4, 5, 7],                 # profondeur des arbres
    'learning_rate': [0.03, 0.05, 0.1], # rythme d'apprentissage
    'l2_leaf_reg': [1, 3, 9],           # régularisation L2 des feuilles
    'iterations': [400]                 # nombre d'arbres
}

cat_base = CatBoostRegressor(random_state=42, verbose=False)

#validation croisée groupée par essai, sinon le score est trop optimiste
cv_groupe = GroupKFold(n_splits=5)

print("Recherche des meilleurs paramètres CatBoost en cours (peut prendre quelques minutes)...")
grid_cat = GridSearchCV(estimator=cat_base, param_grid=param_grid, cv=cv_groupe, scoring='r2', n_jobs=-1)
grid_cat.fit(X_train, y_train, groups=groupes_train)
cat_final = grid_cat.best_estimator_
print(f"Meilleurs paramètres : {grid_cat.best_params_}")
print(f"R2 en validation croisée groupée : {grid_cat.best_score_:.4f}")


In [ ]:
print("--- Évaluation ---")
y_pred = cat_final.predict(X_test)
y_pred = pd.Series(y_pred, index=y_test.index)

# Métriques
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # La racine carrée pour retrouver l'unité physique de E1

print(f"R2   : {r2:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print("\n")

results = pd.DataFrame({'Vraie_Valeur': y_test, 'Prediction_CatBoost': y_pred})
print("Aperçu des prédictions :")
print(results.head())
print("\n")

print("--- Importance des Paramètres ---")
importances = cat_final.feature_importances_
for col, imp in sorted(zip(X_train.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col:<10}: {imp:.1f}%")


In [ ]:
# === Courbes d'apprentissage : R2 et RMSE en fonction de la taille du train set ===
# Même logique que dans AL_modele_1 : CV groupée par essai pour éviter les fuites

from sklearn.base import clone

cat_lc = clone(cat_final)
cv = GroupKFold(n_splits=min(5, groupes_train.nunique()))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
train_sizes = np.linspace(0.1, 1.0, 8)

for ax, scoring, label, is_rmse in [
    (axes[0], "r2", "R2", False),
    (axes[1], "neg_root_mean_squared_error", "RMSE", True),
]:
    sizes, train_scores, val_scores = learning_curve(
        cat_lc, X_train, y_train,
        groups=groupes_train,
        cv=cv, scoring=scoring,
        train_sizes=train_sizes, n_jobs=-1,
    )
    if is_rmse:
        train_scores, val_scores = -train_scores, -val_scores

    tr_m, tr_s = train_scores.mean(1), train_scores.std(1)
    va_m, va_s = val_scores.mean(1), val_scores.std(1)

    ax.plot(sizes, tr_m, "o-", color="tab:blue", label="Train")
    ax.fill_between(sizes, tr_m - tr_s, tr_m + tr_s, alpha=0.15, color="tab:blue")
    ax.plot(sizes, va_m, "o-", color="tab:orange", label="Validation (CV groupée)")
    ax.fill_between(sizes, va_m - va_s, va_m + va_s, alpha=0.15, color="tab:orange")

    ax.set_xlabel("Taille du jeu d'entraînement")
    ax.set_ylabel(label)
    ax.set_title(f"Convergence {label} - CatBoost (E1)")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
